# GPT-2 Inference Engine - Google Colab Setup

This notebook sets up the GPT-2 inference engine on Google Colab with T4 GPU acceleration.

**Runtime:** GPU (T4) | **Estimated time:** 15-20 minutes

## Step 1: Environment Setup

Check GPU availability and install build dependencies.

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install build dependencies
!apt-get update && apt-get install -y build-essential cmake git wget curl htop

In [ ]:
# Check CUDA version
!nvcc --version

## Step 2: Clone and Build GGML with CUDA Support

GGML provides the tensor computation backend with CUDA acceleration.

In [ ]:
# Clone GGML (using master branch for compatibility)
%cd /content
!rm -rf ggml && git clone https://github.com/ggerganov/ggml.git --depth 1 ggml

In [ ]:
# Build GGML with CUDA support
%cd /content/ggml
!rm -rf build && mkdir -p build && cd build && cmake .. -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release && make -j$(nproc)

In [ ]:
# Verify GGML build with CUDA
!cd /content/ggml/build && ls -la src/

## Step 3: Clone Inference Engine

Clone your inference engine source files to the Colab environment.

In [ ]:
# Clone the inference engine repo (includes CMakeLists.txt)
%cd /content
!rm -rf inference-engine && git clone https://github.com/nisbenz/inference-engine.git inference-engine

# Verify CMakeLists.txt exists
!ls -la /content/inference-engine/

In [ ]:
# Upload your source files
# Use the Files panel to upload src/model.cpp, src/model.hpp, src/layers.cpp,
# src/layers.hpp, src/tokenizer.cpp, src/tokenizer.hpp, src/kv_cache.cpp,
# src/kv_cache.hpp, src/main.cpp, src/gguf_loader.cpp, src/gguf_loader.h

# Or run this cell after uploading via the Files panel
import os

src_dir = '/content/inference-engine/src'
print(f'Source files in {src_dir}:')
for f in os.listdir(src_dir):
    print(f'  {f}')

## Step 4: Download GPT-2 Model (GGUF Format)

Download pre-converted GPT-2 GGUF model from HuggingFace.

**Model:** Mungert/gpt2-GGUF (F16 precision for best quality on T4 GPU)
**Config:** 12 layers, 768 hidden, 12 heads, 124M parameters

In [ ]:
# Install HuggingFace Hub for downloading models
!pip install huggingface_hub -q

In [ ]:
# List available GGUF files in the new repository
from huggingface_hub import hf_hub_download
from huggingface_hub import list_repo_files

print("Available GGUF files in Mungert/gpt2-GGUF:")
files = list_repo_files("Mungert/gpt2-GGUF")
for f in files:
    if f.endswith('.gguf'):
        print(f"  {f}")

In [ ]:
# Create model directory
!mkdir -p /content/gpt2-model

# Download GPT-2 F16 model (226 MB) - full precision for best quality
# For T4 GPU with 16GB VRAM, F16 fits easily and gives best results
print("Downloading GPT-2 F16 model...")
model_path = hf_hub_download(
    repo_id="Mungert/gpt2-GGUF",
    filename="f16_q.gguf",
    repo_type="model",
    local_dir="/content/gpt2-model"
)
print(f"Model downloaded to: {model_path}")

In [ ]:
# Download GPT-2 tokenizer files (vocab.json and merges.txt)
# These are required for BPE tokenization
print("Downloading tokenizer files...")
!wget https://raw.githubusercontent.com/openai/gpt-2/master/models/124M/vocab.json -O /content/gpt2-model/vocab.json
!wget https://raw.githubusercontent.com/openai/gpt-2/master/models/124M/merges.txt -O /content/gpt2-model/merges.txt

# Verify all model files
!ls -lh /content/gpt2-model/

In [ ]:
# Update model path in main.cpp
# This ensures the correct model file is loaded
import re

main_cpp_path = '/content/inference-engine/src/main.cpp'
with open(main_cpp_path, 'r') as f:
    content = f.read()

# Update weights path to use F16 model
content = re.sub(
    r'std::string weights_path = "[^"]*"',
    'std::string weights_path = "/content/gpt2-model/f16_q.gguf"',
    content
)

# Update tokenizer paths
content = re.sub(
    r'std::string vocab_path = "[^"]*"',
    'std::string vocab_path = "/content/gpt2-model/vocab.json"',
    content
)
content = re.sub(
    r'std::string merges_path = "[^"]*"',
    'std::string merges_path = "/content/gpt2-model/merges.txt"',
    content
)

with open(main_cpp_path, 'w') as f:
    f.write(content)

print("Updated main.cpp with correct file paths:")
!grep -n "weights_path\|vocab_path\|merges_path" /content/inference-engine/src/main.cpp

## Step 5: Build Custom Inference Engine

Build your own inference engine with the updated source code.

In [ ]:
# Build the inference engine
# Note: GGML_DIR must point to the ggml source directory so cmake can find ggml/ggml.h
!cd /content/inference-engine && mkdir -p build && cd build && cmake .. -DCMAKE_CUDA_PATH=/usr/local/cuda -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_DIR=/content/ggml && make -j$(nproc)

In [ ]:
# Verify build output
!cd /content/inference-engine/build && ls -la bin/

## Step 6: Run Inference

Execute your inference engine with GPU acceleration.

**Usage:** `./gpt2 "prompt" [max_tokens] [temperature] [top_k]`

In [ ]:
# Run with your inference engine
!cd /content/inference-engine/build && ./bin/gpt2 "Once upon a time" 50 0.8 40

## Troubleshooting

### Common Issues:

1. **Build fails**: Check that all source files are uploaded including gguf_loader.cpp and gguf_loader.h
2. **CUDA out of memory**: Use Q8_0 or Q6_K quantization instead of F16
3. **GGUF download fails**: Check HuggingFace connection
4. **Tokenizer fails**: Ensure vocab.json and merges.txt are in /content/gpt2-model/
5. **Tensor mismatch**: Verify model architecture matches (12 layers, 768 hidden, 12 heads)

In [ ]:
# Check GGUF model metadata
import struct

def read_gguf_header(path):
    with open(path, 'rb') as f:
        magic = struct.unpack('I', f.read(4))[0]
        print(f'Magic: {hex(magic)} (should be 0x46554747)')
        version = struct.unpack('I', f.read(4))[0]
        print(f'Version: {version}')
        tensor_count = struct.unpack('Q', f.read(8))[0]
        print(f'Tensor count: {tensor_count}')

read_gguf_header('/content/gpt2-model/f16_q.gguf')

## File Summary

After setup, the structure should be:
```
/content/
├── ggml/                    # GGML library (CUDA enabled)
│   └── build/src/           # Built libraries (libggml.so, libggml-cuda.so, etc.)
├── inference-engine/        # Your project
│   ├── src/
│   │   ├── model.cpp       # GPT-2 model implementation
│   │   ├── model.hpp
│   │   ├── layers.cpp      # Transformer layers (Attention, FFN, etc.)
│   │   ├── layers.hpp
│   │   ├── tokenizer.cpp   # BPE tokenizer
│   │   ├── tokenizer.hpp
│   │   ├── kv_cache.cpp    # KV cache for autoregressive generation
│   │   ├── kv_cache.hpp
│   │   ├── gguf_loader.cpp # GGUF file format loader
│   │   ├── gguf_loader.h
│   │   └── main.cpp        # Entry point
│   ├── build/
│   │   └── bin/gpt2        # Compiled executable
│   └── CMakeLists.txt
└── gpt2-model/
    ├── f16_q.gguf           # GPT-2 model (226 MB)
    ├── vocab.json           # BPE vocabulary
    └── merges.txt           # BPE merge rules
```

## Project Overview

This is a **custom GPT-2 inference engine** built from scratch using:

### Core Components:
1. **GGML Backend** - Tensor computation with CUDA GPU acceleration
2. **Transformer Layers** - Multi-head attention + FFN with residual connections
3. **KV Cache** - Efficient autoregressive generation by caching K/V tensors
4. **BPE Tokenizer** - Byte-pair encoding for GPT-2 text encoding/decoding
5. **GGUF Loader** - Loads pre-converted model weights from HuggingFace

### Architecture:
- **Model:** GPT-2 Base (124M parameters)
- **Layers:** 12 transformer blocks
- **Heads:** 12 attention heads (64 dim each)
- **Hidden:** 768 dimensions
- **FFN:** 3072 intermediate dimensions
- **Context:** 1024 tokens max

### Key Files:
- `model.cpp/hpp` - Main model class, GGUF loading, forward pass
- `layers.cpp/hpp` - Attention, FFN, LayerNorm, RMSNorm, GELU
- `kv_cache.cpp/hpp` - KV cache management for efficient generation
- `tokenizer.cpp/hpp` - GPT-2 BPE tokenization
- `gguf_loader.cpp/hpp` - GGUF file format parsing
- `main.cpp` - CLI entry point

### GPU Acceleration:
- Uses GGML's CUDA backend for tensor operations
- Automatically falls back to CPU if GPU unavailable
- F16 precision model for optimal quality/performance balance